In [50]:
%matplotlib widget
%matplotlib widget
import os
from pathlib import Path
import time
import torch
import numpy as np
import math
import gc
from functools import partial
from dataset_alt import Dataset, load_dataframes_from_folder, reverse_normalization
from torch.utils.data import DataLoader
from transformer_zerostep import GPTConfig, GPT, warmup_cosine_lr, MLP, CausalSelfAttention, LayerNorm, Block
import argparse
import warnings
import matplotlib.pyplot as plt
import copy
import pickle as pkl
from collections import OrderedDict


# set figure parameters
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "cm"
plt.rcParams['axes.labelsize']=14
plt.rcParams['xtick.labelsize']=11
plt.rcParams['ytick.labelsize']=11
plt.rcParams['axes.grid']=True
plt.rcParams['axes.xmargin']=0

In [51]:
model_args = dict(n_layer=8, n_head=4, n_embd=16, n_x=1, n_y=1, n_u=8, block_size=10,
                      bias=False, dropout=0)  

gptconf = GPTConfig(**model_args)

In [52]:
## causal self attention

CSA = CausalSelfAttention(gptconf)
print(CSA)


CausalSelfAttention(
  (c_attn): Linear(in_features=16, out_features=48, bias=False)
  (c_proj): Linear(in_features=16, out_features=16, bias=False)
  (attn_dropout): Dropout(p=0, inplace=False)
  (resid_dropout): Dropout(p=0, inplace=False)
)


In [53]:

test_input_CSA = torch.ones([1,10,16])
test_output_CSA = CSA(test_input_CSA)
# print(output)

isinstance(CSA, torch.nn.DataParallel)
model_weights = CSA.state_dict()
# print(model_weights.keys())
# print(model_weights['c_attn.weight'])

print(model_weights.keys())
gptconf_ord_dict = OrderedDict(gptconf.__dict__)
model_weights.update(gptconf_ord_dict)
print(model_weights.keys())


in_out_dict = {'test_input': test_input_CSA, 'test_output': test_output_CSA}
model_weights.update(in_out_dict)


data_to_save = model_weights
with open("test_CSA.pkl",'wb') as f:
    pkl.dump(data_to_save, f)


odict_keys(['c_attn.weight', 'c_proj.weight'])
odict_keys(['c_attn.weight', 'c_proj.weight', 'block_size', 'n_layer', 'n_head', 'n_embd', 'n_x', 'n_u', 'n_y', 'dropout', 'bias'])


In [54]:
for i in range(16):
    print(test_output_CSA[:,:,i].detach().numpy())

[[0.01897565 0.01897565 0.01897565 0.01897565 0.01897565 0.01897565
  0.01897563 0.01897565 0.01897566 0.01897565]]
[[0.141355   0.141355   0.141355   0.141355   0.141355   0.141355
  0.14135496 0.141355   0.14135504 0.141355  ]]
[[-0.19886176 -0.19886176 -0.19886176 -0.19886176 -0.19886176 -0.19886176
  -0.1988618  -0.19886176 -0.19886182 -0.19886176]]
[[0.0399838  0.0399838  0.0399838  0.0399838  0.03998379 0.0399838
  0.03998379 0.0399838  0.0399838  0.03998379]]
[[0.4815571  0.4815571  0.4815571  0.4815571  0.4815571  0.4815571
  0.48155716 0.4815571  0.4815571  0.4815571 ]]
[[-0.09786152 -0.09786152 -0.09786152 -0.09786152 -0.09786154 -0.09786152
  -0.09786154 -0.09786152 -0.09786154 -0.09786154]]
[[0.02457317 0.02457317 0.02457317 0.02457317 0.02457317 0.02457317
  0.02457315 0.02457317 0.02457317 0.02457317]]
[[-0.25427002 -0.25427002 -0.25427002 -0.25427002 -0.25427002 -0.25427002
  -0.25427002 -0.25427002 -0.25427002 -0.25427002]]
[[-0.13077612 -0.13077612 -0.13077612 -0.13077

In [55]:
print(model_weights.keys())
gptconf_ord_dict = OrderedDict(gptconf.__dict__)
model_weights.update(gptconf_ord_dict)
print(model_weights.keys())

odict_keys(['c_attn.weight', 'c_proj.weight', 'block_size', 'n_layer', 'n_head', 'n_embd', 'n_x', 'n_u', 'n_y', 'dropout', 'bias', 'test_input', 'test_output'])
odict_keys(['c_attn.weight', 'c_proj.weight', 'block_size', 'n_layer', 'n_head', 'n_embd', 'n_x', 'n_u', 'n_y', 'dropout', 'bias', 'test_input', 'test_output'])


In [56]:
MLP_node = MLP(gptconf)
print(MLP_node)

MLP(
  (c_fc): Linear(in_features=16, out_features=64, bias=False)
  (gelu): GELU(approximate='none')
  (c_proj): Linear(in_features=64, out_features=16, bias=False)
  (dropout): Dropout(p=0, inplace=False)
)


In [57]:
test_input = torch.rand([5,10,16])
test_output = MLP_node(test_input)
print(test_output.size())


torch.Size([5, 10, 16])


In [58]:
print(isinstance(MLP_node, torch.nn.DataParallel))
model_weights = MLP_node.state_dict()
model_weights.update(gptconf_ord_dict)

in_out_dict = {'test_input': test_input, 'test_output': test_output}
model_weights.update(in_out_dict)
print(model_weights.keys())

False
odict_keys(['c_fc.weight', 'c_proj.weight', 'block_size', 'n_layer', 'n_head', 'n_embd', 'n_x', 'n_u', 'n_y', 'dropout', 'bias', 'test_input', 'test_output'])


In [59]:

data_to_save = model_weights
with open("test_MLP.pkl",'wb') as f:
    pkl.dump(data_to_save, f)


In [60]:
for i in range(16):
    print(test_output[:,:,i].detach().numpy())

[[-0.08896364 -0.09633137 -0.10059941 -0.06731981  0.00787653 -0.14462754
  -0.09825842 -0.11013401 -0.10782564 -0.1737648 ]
 [-0.01196303 -0.11466187 -0.08170268  0.04570435  0.00715644 -0.06470434
  -0.08175235 -0.17152163 -0.05311309 -0.04908077]
 [-0.0768462  -0.10724816 -0.1308854  -0.1478162   0.00694827 -0.12020266
  -0.00949479 -0.09446344 -0.14174543 -0.01808449]
 [-0.12968701 -0.20656157 -0.02936589 -0.14192605 -0.1419302  -0.08235506
   0.00663712 -0.0630656  -0.03020193  0.04789944]
 [-0.13447739 -0.03880584 -0.07561563 -0.02683392 -0.22168443 -0.06815025
  -0.07943044 -0.00685449 -0.11317906 -0.12055109]]
[[0.1922366  0.24503408 0.11467472 0.21155865 0.12829961 0.12389091
  0.25592238 0.09481052 0.1944372  0.14956117]
 [0.01964315 0.2410882  0.14680274 0.05293486 0.16178176 0.19680554
  0.19182633 0.18955831 0.11080024 0.20680736]
 [0.1963219  0.10025144 0.10778239 0.26351482 0.06353442 0.23918007
  0.13205034 0.24276987 0.28161153 0.09331466]
 [0.14531675 0.271652   0.225

In [61]:
LN = LayerNorm(gptconf.n_embd, gptconf.bias)

perturb = torch.nn.Parameter(torch.ones_like(LN.weight)*0.8 + torch.rand_like(LN.weight)*0.4)
LN.weight = perturb

test_input_LN = torch.rand([5,10,16])

test_output_LN = LN(test_input_LN)

model_weights_LN = LN.state_dict()
model_weights_LN.update(gptconf_ord_dict)

in_out_dict = {'test_input': test_input_LN, 'test_output': test_output_LN}
model_weights_LN.update(in_out_dict)

with open("test_LN.pkl",'wb') as f:
    pkl.dump(model_weights_LN, f)


In [62]:
B = Block(gptconf)
perturb1 = torch.nn.Parameter(torch.ones_like(B.ln_1.weight)*0.8 + torch.rand_like(B.ln_1.weight)*0.4)
perturb2 = torch.nn.Parameter(torch.ones_like(B.ln_2.weight)*0.8 + torch.rand_like(B.ln_2.weight)*0.4)
B.ln_1.weight = perturb1
B.ln_2.weight = perturb2
# B.attn.c_attn.weight
# B.mlp.c_fc.weight

test_input_B = torch.rand([5,10,16])

test_output_B = B(test_input_B)

model_weights_B = B.state_dict()
model_weights_B.update(gptconf_ord_dict)

in_out_dict = {'test_input': test_input_B, 'test_output': test_output_B}
model_weights_B.update(in_out_dict)

with open("test_B.pkl",'wb') as f:
    pkl.dump(model_weights_B, f)


In [63]:
print(test_output_B)

tensor([[[ 9.1566e-01,  8.9762e-01,  6.5076e-01,  2.8548e-01,  1.7052e+00,
           8.1909e-01,  5.8061e-01,  1.1732e+00,  9.4979e-01, -3.7187e-01,
          -5.0592e-01, -6.1941e-02,  1.1203e+00,  7.5958e-02,  2.3013e-01,
           5.8898e-02],
         [ 8.5842e-01,  1.3875e+00,  9.5485e-01, -7.0074e-02,  1.7074e+00,
           3.8149e-01,  1.0250e+00,  1.0532e+00,  1.0724e+00, -1.8766e-01,
          -1.7807e-01, -4.1928e-01,  4.0030e-01, -1.5893e-01,  5.3498e-01,
          -6.1370e-03],
         [ 8.9868e-01,  6.4018e-02,  4.7289e-02,  1.1374e+00,  1.2039e+00,
           2.7771e-01,  4.2368e-01,  1.0788e+00,  9.9031e-02,  1.0220e-01,
           4.2316e-01, -4.4387e-01,  1.0014e+00,  9.7725e-02,  6.5142e-01,
           1.8159e-01],
         [ 9.6384e-01,  4.1829e-01,  8.5815e-01,  1.0133e+00,  2.9264e-01,
           3.1762e-01,  4.2971e-01,  1.0605e+00,  7.9109e-01,  5.8538e-01,
           5.5214e-01, -1.6921e-02,  4.7304e-01, -1.2720e-01,  3.5009e-01,
           6.5825e-01],
    